# AIC 2026 Frame Extraction Worker 1

Official resumable worker for 73 assigned videos. Select **GPU T4 x2**, enable Internet, attach `lyduchoang/aic-26-video`, and enable Kaggle Secrets `AIC_RCLONE_CONFIG` plus `AIC_GDRIVE_FOLDER_ID`. Completed videos are verified on Drive and skipped before TransNetV2 is downloaded.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

SESSION_STARTED_AT = time.time()
REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run_streamed(command, *, cwd=None, env=None, prefix=""):
    print("$", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    lines = []
    for line in process.stdout:
        print(f"{prefix}{line}", end="", flush=True)
        lines.append(line.rstrip())
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Command failed with code {return_code}: {' '.join(map(str, command))}\n"
            + "\n".join(lines[-80:])
        )
    return lines

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {github_token}",
    })
print(f"[repo] phase=sync result=attempting branch={BRANCH}", flush=True)
if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repository: {TARGET}")
if not TARGET.exists():
    run_streamed(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run_streamed(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)
os.chdir(TARGET)
src_path = str((TARGET / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"[repo] phase=sync result=success branch={branch} commit={commit}", flush=True)


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import torch

WORKER_ID = "frame_extraction_worker_1"
VIDEO_IDS = [
    "L21_V001", "L21_V002", "L21_V003", "L21_V005", "L21_V006",
    "L21_V007", "L21_V008", "L21_V009", "L21_V010", "L21_V011",
    "L21_V012", "L21_V013", "L21_V014", "L21_V015", "L21_V016",
    "L21_V017", "L21_V018", "L21_V019", "L21_V021", "L21_V022",
    "L21_V023", "L21_V024", "L21_V025", "L21_V026", "L21_V027",
    "L21_V028", "L21_V029", "L21_V030", "L21_V031",
    "L22_V001", "L22_V002", "L22_V003", "L22_V004", "L22_V005",
    "L22_V006", "L22_V007", "L22_V008", "L22_V009", "L22_V010",
    "L22_V011", "L22_V012", "L22_V013", "L22_V014", "L22_V015",
    "L22_V016", "L22_V017", "L22_V018", "L22_V019", "L22_V020",
    "L22_V021", "L22_V022", "L22_V023", "L22_V024", "L22_V025",
    "L22_V026", "L22_V027", "L22_V028", "L22_V029", "L22_V030",
    "L22_V031",
    "L23_V001", "L23_V002", "L23_V003", "L23_V004", "L23_V005",
    "L23_V006", "L23_V007", "L23_V008", "L23_V009", "L23_V010",
    "L23_V011", "L23_V012", "L23_V013",
]
if len(VIDEO_IDS) != 73 or len(set(VIDEO_IDS)) != 73:
    raise AssertionError(f"Worker assignment must contain 73 unique IDs, got {len(VIDEO_IDS)}")
for excluded in ("L21_V004", "L21_V020", "L24_V001", "L24_V034", "L26_V417"):
    if excluded in VIDEO_IDS:
        raise AssertionError(f"Missing corpus ID was assigned: {excluded}")

ARTIFACT_ROOT = Path("/kaggle/working/aic2026-artifacts")
PACKAGE_NAME = "self-cut-btc-compatible"
PACKAGE_ROOT = ARTIFACT_ROOT / "exports" / PACKAGE_NAME
CONFIG_PATH = Path("configs/offline/frame_extraction.yaml").resolve()
RUNTIME_DIR = Path("/kaggle/working/frame-worker-1")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENT_PATH = RUNTIME_DIR / "assignment.json"
VIDEO_INDEX_PATH = RUNTIME_DIR / "video_index.json"
IDENTITY_DIR = RUNTIME_DIR / "identities"
IDENTITY_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16
WEIGHTS_SHA256 = "a313d0b3bebfa9a71914b375bfdf918a30b5c3b1e6be51972d35dd8078b442de"

print(f"[preflight] phase=assignment count={len(VIDEO_IDS)} first={VIDEO_IDS[0]} last={VIDEO_IDS[-1]}", flush=True)
for binary in ("git", "ffmpeg", "ffprobe", "nvidia-smi"):
    resolved = shutil.which(binary)
    print(f"[preflight] binary={binary} path={resolved or '<missing>'}", flush=True)
    if not resolved:
        raise RuntimeError(f"Required binary is missing: {binary}")
print(f"[preflight] torch={torch.__version__} cuda={torch.cuda.is_available()} gpu_count={torch.cuda.device_count()}", flush=True)
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("Worker 1 requires Kaggle Accelerator = GPU T4 x2")
for gpu_index in range(2):
    gpu_name = torch.cuda.get_device_name(gpu_index)
    print(f"[preflight] gpu={gpu_index} name={gpu_name}", flush=True)
    if "T4" not in gpu_name.upper():
        raise RuntimeError(f"GPU {gpu_index} is not a Tesla T4: {gpu_name}")

VIDEO_SEARCH_ROOT = Path("/kaggle/input/datasets/lyduchoang/aic-26-video/Video")
print(f"[inventory] phase=scan root={VIDEO_SEARCH_ROOT}", flush=True)
if not VIDEO_SEARCH_ROOT.is_dir():
    raise FileNotFoundError(f"Raw-video dataset root is missing: {VIDEO_SEARCH_ROOT}")
wanted = set(VIDEO_IDS)
matches = {}
duplicates = {}
for path in VIDEO_SEARCH_ROOT.rglob("*.mp4"):
    if path.stem not in wanted:
        continue
    if path.stem in matches:
        duplicates.setdefault(path.stem, [matches[path.stem]]).append(path)
    else:
        matches[path.stem] = path
missing = [video_id for video_id in VIDEO_IDS if video_id not in matches]
if missing or duplicates:
    raise RuntimeError(f"Inventory mismatch missing={missing} duplicates={duplicates}")
VIDEO_PATHS = {video_id: matches[video_id] for video_id in VIDEO_IDS}
for index, video_id in enumerate(VIDEO_IDS, start=1):
    print(f"[inventory] {index:02d}/73 video_id={video_id} path={VIDEO_PATHS[video_id]}", flush=True)
ASSIGNMENT_PATH.write_text(json.dumps({"worker_id": WORKER_ID, "video_ids": VIDEO_IDS}, indent=2) + "\n", encoding="utf-8")
VIDEO_INDEX_PATH.write_text(json.dumps({key: str(value) for key, value in VIDEO_PATHS.items()}, indent=2) + "\n", encoding="utf-8")
config_sha256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
for video_id, video_path in VIDEO_PATHS.items():
    identity = {
        "contract_version": "frame-extraction-worker-v1",
        "video_id": video_id,
        "source_size_bytes": video_path.stat().st_size,
        "config_sha256": config_sha256,
        "checkpoint_sha256": WEIGHTS_SHA256,
        "batch_size": BATCH_SIZE,
        "sampling_policy": "lt2-midpoint__2-lt4-quarter-pair__gte4-1.5s__gte7-cap10",
        "extraction_method": "frame-index-select",
    }
    (IDENTITY_DIR / f"{video_id}.json").write_text(json.dumps(identity, sort_keys=True) + "\n", encoding="utf-8")
print("[preflight] phase=inputs result=success count=73", flush=True)


In [ ]:
import base64
import binascii
import configparser
import json
import os
import sys
import urllib.request
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
print("[rclone_preflight] phase=kaggle_secrets result=attempting", flush=True)
rclone_secret = (secrets.get_secret("AIC_RCLONE_CONFIG") or "").strip()
GDRIVE_FOLDER_ID = (secrets.get_secret("AIC_GDRIVE_FOLDER_ID") or "").strip()
if not rclone_secret or not GDRIVE_FOLDER_ID:
    raise ValueError("Enable non-empty AIC_RCLONE_CONFIG and AIC_GDRIVE_FOLDER_ID secrets")
print("[rclone_preflight] phase=kaggle_secrets result=success values=<redacted>", flush=True)
if rclone_secret.lstrip().startswith("["):
    rclone_text = rclone_secret
else:
    encoded = rclone_secret.removeprefix("base64:")
    try:
        rclone_text = base64.b64decode("".join(encoded.split()), validate=True).decode("utf-8")
    except (binascii.Error, UnicodeDecodeError, ValueError) as error:
        raise ValueError("AIC_RCLONE_CONFIG must be rclone.conf text or its base64 payload") from error
parsed = configparser.RawConfigParser()
parsed.read_string(rclone_text)
RCLONE_REMOTE = os.environ.get("AIC_RCLONE_REMOTE", "gdrive")
if not parsed.has_section(RCLONE_REMOTE) or parsed.get(RCLONE_REMOTE, "type", fallback="") != "drive":
    raise ValueError(f"rclone config must contain a [{RCLONE_REMOTE}] drive remote")
RCLONE_CONFIG = Path("/tmp/aic2026-rclone/rclone.conf")
RCLONE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
RCLONE_CONFIG.write_text(rclone_text, encoding="utf-8")
RCLONE_CONFIG.chmod(0o600)
RCLONE_BIN = Path("/kaggle/working/bin/rclone")
if not RCLONE_BIN.is_file():
    archive = Path("/tmp/rclone.zip")
    print("[rclone_preflight] phase=install result=attempting", flush=True)
    urllib.request.urlretrieve("https://downloads.rclone.org/rclone-current-linux-amd64.zip", archive)
    with zipfile.ZipFile(archive) as bundle:
        member = next(item for item in bundle.infolist() if item.filename.endswith("/rclone"))
        RCLONE_BIN.parent.mkdir(parents=True, exist_ok=True)
        RCLONE_BIN.write_bytes(bundle.read(member))
    RCLONE_BIN.chmod(0o755)
print(f"[rclone_preflight] phase=install result=success path={RCLONE_BIN}", flush=True)

SYNC_BASE = [
    sys.executable, "-u", "scripts/sync_frame_video_with_rclone.py",
    "--rclone-bin", str(RCLONE_BIN), "--config", str(RCLONE_CONFIG),
    "--remote", RCLONE_REMOTE, "--root-folder-id", GDRIVE_FOLDER_ID,
    "--remote-root-name", PACKAGE_NAME, "--worker-id", WORKER_ID,
]
preflight_command = SYNC_BASE[:3] + ["preflight"] + SYNC_BASE[3:] + ["--attempts", "5"]
preflight_lines = run_streamed(preflight_command, prefix="[drive] ")
PREFLIGHT_REPORT = json.loads(next(line for line in reversed(preflight_lines) if line.startswith("{")))
print(f"[rclone_preflight] phase=drive result=success folder_url={PREFLIGHT_REPORT.get('folder_url')}", flush=True)
scan_command = SYNC_BASE[:3] + ["scan"] + SYNC_BASE[3:] + ["--identity-dir", str(IDENTITY_DIR), "--attempts", "5"]
for video_id in VIDEO_IDS:
    scan_command.extend(["--video-id", video_id])
scan_lines = run_streamed(scan_command, prefix="[drive_scan] ")
INITIAL_SCAN = json.loads(next(line for line in reversed(scan_lines) if line.startswith("{")))
PENDING_VIDEO_IDS = INITIAL_SCAN["pending"]
print(f"[drive_scan] completed={len(INITIAL_SCAN['completed'])} pending={len(PENDING_VIDEO_IDS)}", flush=True)
print(f"[drive_scan] pending_ids={PENDING_VIDEO_IDS}", flush=True)


In [ ]:
import hashlib
import os
import urllib.request
from pathlib import Path

TRANSNET_SOURCE = Path("/kaggle/working/TransNetV2-source")
PYTORCH_MODULE = TRANSNET_SOURCE / "inference-pytorch" / "transnetv2_pytorch.py"
WEIGHTS = Path("/kaggle/working/transnetv2-pytorch/transnetv2-pytorch-weights.pth")
WEIGHTS_URL = "https://huggingface.co/ByteDance/shot2story/resolve/ff853c571fd92eb4e0c5713e27f2a323ac903f67/transnetv2-pytorch-weights.pth?download=true"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not PENDING_VIDEO_IDS:
    print("[transnet_setup] result=skipped reason=all_assigned_videos_complete", flush=True)
else:
    print(f"[transnet_setup] result=attempting pending={len(PENDING_VIDEO_IDS)}", flush=True)
    if not PYTORCH_MODULE.is_file():
        if TRANSNET_SOURCE.exists():
            raise RuntimeError(f"TransNet source exists but PyTorch module is missing: {TRANSNET_SOURCE}")
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run_streamed(["git", "clone", "--depth=1", "https://github.com/soCzech/TransNetV2.git", str(TRANSNET_SOURCE)], env=clone_env)
    if not WEIGHTS.is_file() or sha256(WEIGHTS) != WEIGHTS_SHA256:
        WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
        partial = WEIGHTS.with_suffix(".partial")
        print("[transnet_setup] checkpoint=downloading", flush=True)
        urllib.request.urlretrieve(WEIGHTS_URL, partial)
        if sha256(partial) != WEIGHTS_SHA256:
            raise ValueError("Downloaded TransNetV2 checkpoint SHA-256 mismatch")
        os.replace(partial, WEIGHTS)
    print(f"[transnet_setup] result=success module={PYTORCH_MODULE} checkpoint_sha256=verified", flush=True)


In [ ]:
import json
import sys

if not PENDING_VIDEO_IDS:
    WORKER_REPORT = {
        "status": "completed", "worker_id": WORKER_ID,
        "remote_completed": INITIAL_SCAN["completed"], "remaining": [],
    }
else:
    worker_command = [
        sys.executable, "-u", "scripts/run_frame_extraction_worker.py",
        "--worker-id", WORKER_ID,
        "--assignment", str(ASSIGNMENT_PATH),
        "--video-index", str(VIDEO_INDEX_PATH),
        "--identity-dir", str(IDENTITY_DIR),
        "--config", str(CONFIG_PATH),
        "--output-root", str(ARTIFACT_ROOT),
        "--package-root", str(PACKAGE_ROOT),
        "--entrypoint", str(PYTORCH_MODULE),
        "--weights", str(WEIGHTS),
        "--batch-size", str(BATCH_SIZE),
        "--rclone-bin", str(RCLONE_BIN),
        "--rclone-config", str(RCLONE_CONFIG),
        "--rclone-remote", RCLONE_REMOTE,
        "--drive-root-folder-id", GDRIVE_FOLDER_ID,
        "--remote-root-name", PACKAGE_NAME,
        "--session-start-epoch", str(SESSION_STARTED_AT),
        "--accept-new-work-seconds", str(11 * 3600 + 15 * 60),
        "--scan-attempts", "5",
        "--upload-attempts", "10",
    ]
    worker_lines = run_streamed(worker_command, prefix="[worker_1] ")
    WORKER_REPORT = json.loads(next(line for line in reversed(worker_lines) if line.startswith("{")))
print(json.dumps(WORKER_REPORT, indent=2), flush=True)


In [ ]:
print(f"[final] worker_id={WORKER_ID}", flush=True)
print(f"[final] status={WORKER_REPORT['status']}", flush=True)
print(f"[final] completed={len(WORKER_REPORT.get('remote_completed', []))}/73", flush=True)
print(f"[final] remaining={WORKER_REPORT.get('remaining', [])}", flush=True)
print(f"[final] drive_url={PREFLIGHT_REPORT.get('folder_url')}", flush=True)
print(f"[final] benchmark_remote={PACKAGE_NAME}/benchmark/{WORKER_ID}.json", flush=True)
print("[final] Rerun Save Version to continue only the remaining videos.", flush=True)
